# Tutoriel URBasic

limit + feedback => safety

### Set the comunication with the robot

In [ ]:
import sys
import os

import numpy as np
from time import sleep
sys.path.append(os.path.abspath(".."))

In [ ]:
from URBasic import ISCoin
from URBasic import Joint6D
from URBasic import TCP6D
from URBasic import TCP6DDescriptor

robot_ip = "10.30.5.159"
robot_opened_gripper_size_mm = 40. # Max oppenning of gripper

iscoin = ISCoin(robot_ip, robot_opened_gripper_size_mm)

if not iscoin.isInitOk:
    print("Error robot not initialised")
    sys.exit(1)

iscoin.robot_control.reset_error()

# Base robot position in Joint space
j_o = Joint6D.createFromRadians(
        -1.5708, -1.5708, -1.0472, -2.0944, 1.5708, 0.0
    )

## TCP Calibration

In [ ]:
# initial_pos   = TCP6D.createFromMetersRadians(x, y, z, rx, ry, rz)    # Coordinate of the tools
# initial_joint = Joint6D.createFromRadians(j1, j2, j3, j4, j5, j6)     # Value of the robot joints

robot_arm = iscoin.robot_control

### Collect TCP pivot data

In [ ]:
NUM_MEASURES = 20

tcps = []

robot_arm.set_tcp(TCP6D.createFromMetersRadians(0,0,0,0,0,0))
k = 0
while k < NUM_MEASURES:
    robot_arm.freedrive_mode()
    i = input(f"[{k+1}/{NUM_MEASURES}] Move robot, then press ENTER to capture. (or q to quit)")
    if i == "q":
        if k < 6:
            print("You should give min 6 points")
            continue
        else:
            break

    robot_arm.end_freedrive_mode()

    # Return robot positions
    tcp = robot_arm.get_actual_tcp_pose().toList()

    tcps.append(tcp)
    k += 1
    print("Captured sample", k)


robot_arm.end_freedrive_mode()

In [ ]:
for t in tcps:
    print(t)

### Pivot calibration

In [ ]:
tcps = [
    [0.0024946337740892888, -0.3594913915339868, 0.06033167700558813, -0.62069755529755, -2.369727486212462, -0.13752393611863362],
    [-0.07482225018547103, -0.35685269130655106, 0.07009337653443365, 1.9372057315922462, 2.190232377619723, 0.6719840297817087],
    [-0.08776793211801745, -0.2979812165503483, 0.051154992555969794, 2.3781388566898687, 0.2623998383341909, 0.7178051284770465],
    [-0.0006933205535983589, -0.340924257935631, 0.06551204074775754, -2.879088789274111, -0.4716877213448937, 0.8276834472055694],
    [-0.04094004360899719, -0.3437262194580261, 0.0786140415986546, -3.087426800813378, 0.2627406174702693, 0.06421242457662117],
    [0.026621421400990636, -0.3469050147213359, 0.033956680032955974, -2.5014091646201537, -0.22918589994059693, 1.5168173293250897],
]

In [ ]:
def pose_to_T(pose):
    """
    pose: [x, y, z, rx, ry, rz] (UR standard, base_T_flange)
    returns 4x4 homogeneous matrix
    """
    x, y, z, rx, ry, rz = pose
    theta = np.linalg.norm([rx, ry, rz])
    if theta < 1e-9:
        R = np.eye(3)
    else:
        k = np.array([rx, ry, rz]) / theta
        K = np.array([[0, -k[2], k[1]],
                      [k[2], 0, -k[0]],
                      [-k[1], k[0], 0]])
        R = np.eye(3) + np.sin(theta)*K + (1-np.cos(theta))*(K @ K)

    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = [x, y, z]
    return T

In [ ]:
def calibrate_tcp(flange_poses):
    """
    flange_poses: list of 4x4 homogeneous matrices T_i (base_T_flange)
    Returns:
        t_tcp_flange: 3-vector, TCP in flange frame
        c_tcp_base:   3-vector, TCP in base frame
    """
    N = len(flange_poses)
    A = np.zeros((3 * N, 6))
    b = np.zeros((3 * N,))

    for i, T in enumerate(flange_poses):
        R = T[:3, :3]
        p = T[:3, 3]

        A[3*i:3*i+3, 0:3] = R          # R_i
        A[3*i:3*i+3, 3:6] = -np.eye(3) # -I
        b[3*i:3*i+3] = -p              # -p_i

    # Least squares solution
    x, residuals, rank, s = np.linalg.lstsq(A, b, rcond=None)
    t_tcp_flange = x[0:3]
    c_tcp_base   = x[3:6]
    return t_tcp_flange, c_tcp_base

In [ ]:
# Convert poses to matrices
Ts = [pose_to_T(p) for p in tcps]

# Extract R_i and p_i
Rs = [T[:3, :3] for T in Ts]
ps = [T[:3, 3] for T in Ts]

# Your calibrated TCP in flange frame
t = np.array([-0.00072928, -0.00040785, 0.07966732])

# Compute estimated TCP position in base for each measurement
c_est = [ps[i] + Rs[i] @ t for i in range(len(ps))]

# Compute spread (should be small)
c_est = np.array(c_est)
spread = np.max(np.linalg.norm(c_est - np.mean(c_est, axis=0), axis=1))

# Compute flange motion (should be large)
flange_motion = np.max(np.linalg.norm(ps - ps[0], axis=1))

# Compute rotation variation (should be large)
rot_angles = []
for R in Rs:
    angle = np.arccos((np.trace(R) - 1) / 2)
    rot_angles.append(angle)
rot_variation = np.max(rot_angles) - np.min(rot_angles)

print("TCP consistency (mm):    \t", spread * 1000)
print("Flange motion (cm):      \t", flange_motion * 100)
print("Rotation variation (deg):\t", np.degrees(rot_variation))


How to interpret the results
1. TCP consistency (mm)
    - Ideally: 0.1–1.0 mm
    - If > 2–3 mm → calibration is noisy or poses were too similar.

2. Flange motion (cm)
    - Should be 10–40 cm depending on your robot.
    - If < 5 cm → robot didn’t move enough.

3. Rotation variation (deg)
    - Should be at least 30–60°
    - If < 10° → calibration becomes ill‑conditioned.

In [ ]:
t_tcp_flange, c_tcp_base = calibrate_tcp(Ts)
print("TCP in flange frame:", t_tcp_flange, " <-")  # Tool offset
print("TCP in tool   frame:",   c_tcp_base)         # Pivot point -> Fixed point in the world in TCP coordinate

In [ ]:
tcp_offset = TCP6D.createFromMetersRadians(t_tcp_flange[0], t_tcp_flange[1], t_tcp_flange[2], 0 , 0, 0)
robot_arm.set_tcp(tcp_offset)

### Validation Calibration

The validation is done visually when the robot approch the same point from diffrent angles.

In [ ]:
ROT = 0.5   # about ~28 degrees

In [ ]:
# Capture reference pose
robot_arm.freedrive_mode()
input("Move robot so the stylus is in the hole, then press ENTER.")
tcp_ref = robot_arm.get_actual_tcp_pose()
robot_arm.end_freedrive_mode()

# Move back to reference pose once
robot_arm.movel(tcp_ref, a=1.2, v=0.25)

In [ ]:
# Helper: apply rotation around TCP
def rotate_around_tcp(tcp: TCP6D, rx, ry, rz):
    new_pose = TCP6D.createFromMetersRadians(
        tcp.x, tcp.y, tcp.z,
        tcp.rx + rx,
        tcp.ry + ry,
        tcp.rz + rz
    )
    return new_pose

In [ ]:
# Perform validation rotations
axes = [
    (ROT, 0, 0),
    (-ROT, 0, 0),
    (0, ROT, 0),
    (0, -ROT, 0),
    (0, 0, ROT),
    (0, 0, -ROT)
]

for rx, ry, rz in axes:
    test_pose = rotate_around_tcp(tcp_ref, rx, ry, rz)
    print(f"Rotating: rx={rx}, ry={ry}, rz={rz}")

    robot_arm.movel(test_pose, a=1.2, v=0.25)
    robot_arm.waitRobotIdleOrStopFlag()
    sleep(2)
    robot_arm.movel(tcp_ref, a=1.2, v=0.25)
    robot_arm.waitRobotIdleOrStopFlag()
    sleep(1)

print("Validation complete.")

### Relative TCP detectionPosition

Now that the robot know the end point of the tool we need to determine where the tool should go in real world.

To do so we compute a transformation which will allow use to convert the world coordinate into TCP coordinate.

In [ ]:
def similarity_transform_3D(A, B):
    """
    Limited to similar transformation
    Compute s, R, t such that:  B ≈ s * R @ A + t
    A: Nx3 points in source frame
    B: Nx3 points in target frame
    """
    A = np.asarray(A)[:, :3]
    B = np.asarray(B)[:, :3]
    assert A.shape == B.shape
    N = A.shape[0]

    centroid_A = A.mean(axis=0)
    centroid_B = B.mean(axis=0)

    AA = A - centroid_A
    BB = B - centroid_B

    H = AA.T @ BB

    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    if np.linalg.det(R) < 0:
        Vt[2, :] *= -1
        R = Vt.T @ U.T

    # scale factor
    var_A = (AA ** 2).sum()
    s = (S.sum()) / var_A

    t = centroid_B - s * R @ centroid_A

    return s, R, t

# Example usage:
# s, R_w2tcp, t_w2tcp = similarity_transform_3D(p_world, p_tcp)

# def world_to_tcp(p):
#     return s * ( R_w2tcp @ p ) + t_w2tcp

In [ ]:
def affine_transform_3D(A, B):
    """
    Solve B ≈ A*M + t  for M (3x3) and t (3,)
    A: Nx3 source points
    B: Nx3 target points
    """
    A = np.asarray(A)[:, :3]
    B = np.asarray(B)[:, :3]
    assert A.shape == B.shape
    N = A.shape[0]

    # Build augmented matrix [A | 1]
    A_aug = np.hstack([A, np.ones((N, 1))])   # Nx4

    # Solve least squares: A_aug @ X = B
    # X is 4x3 → last row is translation
    X, _, _, _ = np.linalg.lstsq(A_aug, B, rcond=None)

    M = X[:3, :].T   # 3x3
    t = X[3, :]      # 3,
    return M, t.tolist()

# Example usage:
# M, t = affine_transform_3D(p_world, p_tcp[:, :3])

# def world_to_tcp(p):
#     return M @ p + t


In [ ]:
p_tcp = []

p_world = np.array([
    [  0,   0, 0],
    [ 10,   0, 0],
    [ 10,  40, 0],
])
pw = 0
while pw < len(p_world):
    robot_arm.freedrive_mode()
    input(f"[{pw+1}/{len(p_world)}] Move robot, then press ENTER to capture.")

    robot_arm.end_freedrive_mode()

    # Return robot positions
    tcp = robot_arm.get_actual_tcp_pose().toList()

    print(tcp)
    while True:
        i = input("Do you confirme the mesure? y/n")
        if i == "y":
            p_tcp.append(tcp)
            pw += 1
            break
        elif i == "n":
            break

p_tcp = np.array(p_tcp)

robot_arm.end_freedrive_mode()

In [ ]:
# p_tcp = np.array([
#     [0.2000, -0.1000, 0.2500, 0, 3.14, 0],
#     [0.2925, -0.0651, 0.2346, 0, 3.14, 0],
#     [0.2361,  0.0378, 0.2891, 0, 3.14, 0],
# ])

# p_world = np.array([
#     [  0,   0, 0],
#     [ 10,   0, 0],
#     [ 10,  40, 0],
# ])

M, t = affine_transform_3D(p_world, p_tcp)

# Convert world -> tcp
def world_to_tcp(p):
    return M @ p + t

In [ ]:
# test: transform tcp[i] -> world and compare with p_world[i]
for i in range(len(p_tcp)):
    predicted = world_to_tcp(p_world[i])
    print("pred:", predicted, " real:", p_tcp[i][:3])
    if not np.allclose(predicted, p_tcp[i][:3], atol=1e-6):
        print("Mismatch at index", i)

In [ ]:
# Test motion comparaison between predicted and real
motion = []
for i in range(len(p_tcp)):
    predicted = world_to_tcp(p_world[i])
    x,y,z = predicted
    t = TCP6D.createFromMetersRadians(x,y,z,0,3.14,0)
    m = TCP6DDescriptor(t)
    motion.append(m)
    x,y,z,rx,ry,rz = p_tcp[i]
    t = TCP6D.createFromMetersRadians(x,y,z,rx,ry,rz)
    m = TCP6DDescriptor(t)
    motion.append(m)

robot_arm.movej(j_o)
robot_arm.movel_waypoints(motion)
robot_arm.movej(j_o)

In [ ]:
robot_arm.get_inverse_kin

## Test motion

It is possible to store robot position in two ways;

1. Tool Center Position(TCP) Position (TCP6D);
    - This position is center on the tool position by encoding the cartesian coordinate (**x**,**y**,**z**) and the orientation along the axis (**rx**,**ry**,**rz**) of the tool.
    - **warning**: Will it make easy to compute the tool position, the robot may have multiple way to reach the say tool position.
    - *Application*: When we need to know the tool position without error.

2. Joint Position (Joint6D);
    - A robot arm is composed of different motor called joint. By setting each of these robot to a specific position the robot can reach a position. In this configuration the robot will have only one possibility to reach the position as it encode the rotation of each joint of the robot.
    - **Warning**: Will it make the position of the robot know without any doubes, it is more difficult to know the end point of the robot -> TCP
    - *Application*: For safe position, and calibration position.



In [ ]:
""" 
movel(self, pose : TCP6D, a : float = 1.2, v : float = 0.25, t : float = 0, r : float = 0, wait : bool = True)
    Move to position (linear in tool-space)
    Parameters:
    pose: target pose [m]
    a:    tool acceleration [m/s^2]
    v:    tool speed [m/s]
    t:    time [S]
    r:    blend radius [m]
    wait: function return when movement is finished

movej(self, joints : Joint6D, a : float = 1.4, v : float = 1.05, t : float = 0, r : float = 0, wait : bool = True)
    Move to position (linear in joint-space)
    Parameters:
    joints:    joint positions [rad]
    a:    joint acceleration of leading axis [rad/s^2]
    v:    joint speed of leading axis [rad/s]
    t:    time [S]
    r:    blend radius [m]
    wait: function return when movement is finished
"""

j_o = Joint6D.createFromRadians(
        -1.5708, -1.5708, -1.0472, -2.0944, 1.5708, 0.0
        # θ_1,    θ_2,     θ_3,     θ_4,    θ_5,    θ_6
)

motion = [
    # 1. Move robot to a safe home position (joint space)
    j_o,

    # 2. Move above the pick location (TCP space)
    TCP6D.createFromMetersRadians(
        0.200, -0.250, 0.300,    # x, y, z in meters
        3.1416, 0.0000, 0.0000   # rx, ry, rz in radians
    ),

    # 3. Move down to the pick point (TCP space)
    TCP6D.createFromMetersRadians(
        0.200, -0.250, 0.180,
        3.1416, 0.0000, 0.0000
    ),

    # 4. Lift back up (TCP space)
    TCP6D.createFromMetersRadians(
        0.200, -0.250, 0.300,
        3.1416, 0.0000, 0.0000
    ),

    # 5. Move to a different joint configuration (joint space)
    Joint6D.createFromRadians(
        1.5708, -1.0472, 1.0472, -1.0472, 1.5708, 1.5708
    ),

    # 6. Move to a place location (TCP space)
    TCP6D.createFromMetersRadians(
        -0.200, -0.250, 0.350,
        3.1416, 0.0000, 0.0000
    ),

    # 7. Return to a safe home position (joint space)
    j_o,
]

for m in motion:
    if isinstance(m, TCP6D):
        robot_arm.movel(m)
    elif isinstance(m, Joint6D):
        robot_arm.movej(m)

## Test gripper

**Important**: Before using the gripper it should be activated.

**Important**: When the gripper will not be used you can deactivat it.

In [ ]:
robot_gripper = iscoin.gripper

robot_gripper.activate()

robot_gripper.waitUntilActive()
if not robot_gripper.isActivated():
  print("Error the gripper was not activated")

In [ ]:
print("Actual gripper opening:\t" + str(robot_opened_gripper_size_mm) + " [mm]")
print("Approximate resolution:\t" + str(robot_opened_gripper_size_mm/255) + " [mm] per unit")

Actual gripper opening: 40 mm  
Approximate resolution: ~0.2 mm per unit

### Basic Command Gripper

In [ ]:
"""
  pos: Position to move to (0-255, 0 being open (50mm), 255 being closed (0mm)). Around 0.2mm per unit.
  speed: Speed of the gripper (0-255, 0 being slowest) default = 255
  force: Force of the gripper (0-255, 0 being weakest) default = 50
"""

if not robot_gripper.move(pos=155, speed=255, force=50): # 20mm -> 255 - (20/0.2) = 155
  print("Failed to move gripper")
sleep(1) # Allow time for the gripper to move

if not robot_gripper.isMoving():
  print("The gripper is still moving")
  robot_gripper.waitUntilStopped() # Alternative to the sleep

if not robot_gripper.open(speed=255, force=50):
  print("Failed to open gripper")
else:
  robot_gripper.waitUntilStopped()
  if not robot_gripper.close(speed=255, force=50):
    print("Failed to close gripper")
robot_gripper.waitUntilStopped()

### Gripper Object Detection

In [ ]:
robot_gripper.close(speed=255, force=50)
robot_gripper.waitUntilStopped()

if not robot_gripper.hasDetectedObject():
    print("The gripper did not catch an object")
else:
    print("The gripper did catch an object")

if robot_gripper.isMoving():
    print("I cannot measure the object dimension as the gripper move.")
else:
    print("The gripper catch an object of dimension " + str(robot_gripper.getEstimatedPositionMm()) + "mm")

sleep(10)

robot_gripper.open(speed=255, force=50)
robot_gripper.waitUntilStopped()

# This will close the gripper and say if the object match the expectations
if not robot_gripper.closeAndCheckSize(expected_size_mm=15., plus_margin_mm=2.5, minus_margin_mm=3.2, speed=255, force=50):
    print("The gripper did not catch the expected object:     " + str(robot_gripper.getEstimatedPositionMm()) + " mm")
else:
    print("The gripper did catch a potential expected object: " + str(robot_gripper.getEstimatedPositionMm()) + " mm")
# robot_gripper.waitUnitilStopped() is incorporated into closeAndCheckSize function

sleep(10)

robot_gripper.open(speed=255, force=50)
robot_gripper.waitUntilStopped()
robot_gripper.close(speed=255, force=50)
robot_gripper.waitUntilStopped()

# This will open the gripper and say if the object match the expectations
if not robot_gripper.openAndCheckSize(expected_size_mm=15., plus_margin_mm=2.5, minus_margin_mm=3.2, speed=255, force=50):
    print("The gripper did not catch the expected object:     " + str(robot_gripper.getEstimatedPositionMm()) + " mm")
else:
    print("The gripper did catch a potential expected object: " + str(robot_gripper.getEstimatedPositionMm()) + " mm")
# robot_gripper.waitUnitilStopped() is incorporated into openAndCheckSize function

sleep(10)

robot_gripper.close(speed=255, force=50)
robot_gripper.waitUntilStopped()

In [ ]:
robot_gripper.move(pos=55, speed=255, force=50) # 40mm -> 255 - (40/0.2) = 55
robot_gripper.waitUntilStopped()
robot_gripper.deactivate()

## Camera

In [ ]:
iscoin.camera.resetCameraSettings()
iscoin.displayCameraStreamOCVParallel() # Display the camera stream

In [ ]:
import cv2
from PIL import Image

from URBasic import CameraSettings
from URBasic import RobotiqWristCamera

In [ ]:
# Plays with camera LEDs
robot_camera = iscoin.camera
cs = CameraSettings()

for i in range(0,11):
    print(f'Setting manual mode to {10*i}')
    cs.lightingSettings.setManualMode(10*i)
    # Actually send changes
    if not robot_camera.setCameraSettings(cs):
        print('Error setting camera settings')
        break
    sleep(0.2)
print('Reset default mode')
cs.lightingSettings.setOffMode()
robot_camera.setCameraSettings(cs)
sleep(3)

img = robot_camera.getImageAsImage(type = RobotiqWristCamera.ImageType.COLOR)

cv2.sho